# Deber — Semana 3: Carga de Datos, Series/DataFrames, loc/iloc y Análisis de Texto
## Análisis de Datos (TDSD353) | Período 2026-A
## ESFOT — Escuela Politécnica Nacional


**Nombre:** __Melva Patricia Suarez Casco___  
**Fecha:** __27/04/2026____
<br/>
---

### Objetivo de este Taller
Aplicar las técnicas aprendidas en clase para:
1. Cargar datos desde archivos **JSON**, **TXT** (CSV) y **XML** usando Pandas.
2. Detectar y documentar **problemas de calidad de datos**.
3. Unificar múltiples DataFrames con `pd.concat()` y exportar a CSV.
4. Comprender y usar **Series** y **DataFrames**.
5. Acceder a datos con **`loc`** (por etiqueta) e **`iloc`** (por posición).
6. Descargar y procesar texto desde la web con `urllib`.

### Archivos de trabajo
| Archivo | Contenido | IDs |
|---------|-----------|-----|
| `productos.json` | 20 productos de librería | 1–20 |
| `productos.txt` | 20 productos de librería (formato CSV) | 21–40 |
| `productos.xml` | 20 productos de librería | 41–60 |
| `netflix.csv` | Catálogo de Netflix (~8800 títulos) | — |

> ⚠️ Los archivos de productos contienen **errores intencionales** (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

---

---
## PARTE A: Pipeline de datos — Inventario de Papelería

Tenemos 3 archivos con datos del inventario de una Papelería:
- `productos.json` — 20 productos (IDs 1–20)
- `productos.txt` — 20 productos (IDs 21–40, separados por coma)
- `productos.xml` — 20 productos (IDs 41–60)

Columnas compartidas: `id`, `nombre`, `precio_compra`, `categoria`, `stock`, `proveedor`, `precio_venta_publico`

> ⚠️ Los datos contienen errores intencionales (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

### A-1. Importar librerías

| Librería | Uso |
|----------|-----|
| `pandas` | Manipulación de datos tabulares |
| `urllib.request` | Descarga de archivos desde la web |
| `collections.Counter` | Conteo de frecuencias |
| `time` | Medición de tiempos |
| `re` | Expresiones regulares para limpieza de texto |

In [221]:
import pandas as pd
import urllib.request
from collections import Counter
import time 
import re
print ('Librerias importadas')
print(f'Version de pandas: {pd.__version__}')

Librerias importadas
Version de pandas: 2.3.3


### A-2. Cargar JSON con `pd.read_json()`

**JSON** (JavaScript Object Notation): formato ligero muy común en APIs web.

```python
df = pd.read_json('archivo.json')
```

Parámetros útiles: `orient` (formato del JSON), `encoding`, `dtype` (forzar tipos).

In [222]:
df_json = pd.read_json('productos.json', dtype=str)

print(f'Forma: {df_json.shape}\t {df_json.shape[0]} filas y {df_json.shape[1]} columnas')

df_json.head()

Forma: (20, 7)	 20 filas y 7 columnas


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,1,Bolígrafo BIC Azul,0.35000000000000003,Escritura,500,DistribuPapel S.A.,0.75
1,2,Cuaderno Universitario 100 hojas,1.8,Cuadernos,300,Norma Ecuador,3.5
2,3,Lápiz HB Faber-Castell,0.2,Escritura,800,DistribuPapel S.A.,0.45
3,4,Resaltador Amarillo,error,Escritura,150,OfficeMax Ecuador,1.2
4,5,Carpeta Manila A4,0.15,Archivo,1000,PapelPro Cía. Ltda.,0.3


### A-3. Cargar TXT con `pd.read_csv()`

**TXT tabular** (datos separados por coma): los archivos `.txt` con formato de tabla se leen con `read_csv()` ajustando el separador.

```python
df = pd.read_csv('archivo.txt', sep=',')   # separador: coma
df = pd.read_csv('archivo.txt', sep='\t')  # separador: tabulación
```

> **Diferencia CSV vs TXT**: ambos son texto plano. La extensión `.txt` no implica un formato fijo — siempre revisa el separador antes de cargar.

In [223]:
df_txt = pd.read_csv('productos.txt', sep=',', dtype=str)
print(f'forma: {df_txt.shape}\t {df_txt.shape[0]} filas y {df_txt.shape[1]} columnas')
df_txt.head()


forma: (20, 7)	 20 filas y 7 columnas


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,21,Sacapuntas Doble Uso,0.15,Escritura,700,DistribuPapel S.A.,0.35
1,22,Cuaderno Espiral A5 80 hojas,1.40,Cuadernos,220,Norma Ecuador,2.80
2,23,Lápiz de Color x24,2.10,Manualidades,160,Norma Ecuador,4.20
3,24,Pegamento en Barra 40g,0.55,Manualidades,430,OfficeMax Ecuador,1.05
4,25,Papel Lustre x10 colores,0.60,Papel,380,PapelPro Cía. Ltda.,1.20


### A-4. Cargar XML con `pd.read_xml()`

**XML** (eXtensible Markup Language): formato jerárquico usado en servicios web SOAP y datos gubernamentales.

```python
df = pd.read_xml('archivo.xml', xpath='.//producto')
```

El parámetro `xpath` indica qué nodos representan cada fila. 

In [224]:
df_xml = pd.read_xml('productos.xml', xpath='.//producto', dtype=str)
print(f'forma: {df_xml.shape}\t {df_xml.shape[0]} filas y {df_xml.shape[1]} columnas')
df_xml.head()

forma: (20, 7)	 20 filas y 7 columnas


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,41,Tiza Blanca Caja x100,1.50,Escritura,180,Norma Ecuador,3.00
1,42,Papelógrafo Cuadriculado x50,2.80,Papel,130,PapelPro Cía. Ltda.,5.20
2,43,Organizador de Escritorio Plástico,3.60,Organización,65,OfficeMax Ecuador,6.80
3,44,Cuaderno Cosido 50 hojas,0.90,Cuadernos,texto,Norma Ecuador,1.80
4,45,Cartulina de Colores x10,0.70,Papel,320,PapelPro Cía. Ltda.,1.40


### A-5. Verificar consistencia y detectar errores

Antes de concatenar, verificamos que los 3 DataFrames tengan las **mismas columnas** y documentamos los problemas de calidad.

Errores comunes en datos reales:
- Valores `null` / `NaN` (datos faltantes)
- Texto donde debería haber números (`"error"`, `"N/A"`, `"texto"`)
- Valores lógicamente inválidos (Ejemplo stock negativo)
- Campos vacíos (`""`)

In [225]:
cols_json = set(df_json.columns)
cols_txt  = set(df_txt.columns)
cols_xml  = set(df_xml.columns)

print('Columnas JSON: ', list(df_json.columns))
print('Columnas TXT : ', list(df_txt.columns))
print('Columnas XML : ', list(df_xml.columns))

print('JSON == TXT :', cols_json == cols_txt)
print('JSON == XML :', cols_json == cols_xml)
print('TXT == XML  :', cols_txt == cols_xml)

Columnas JSON:  ['id', 'nombre', 'precio_compra', 'categoria', 'stock', 'proveedor', 'precio_venta_publico']
Columnas TXT :  ['id', 'nombre', 'precio_compra', 'categoria', 'stock', 'proveedor', 'precio_venta_publico']
Columnas XML :  ['id', 'nombre', 'precio_compra', 'categoria', 'stock', 'proveedor', 'precio_venta_publico']
JSON == TXT : True
JSON == XML : True
TXT == XML  : True


In [226]:
errores = {
    'JSON (IDs 1-20)': [
        "id=4  | precio_compra = 'error'  → texto en campo numérico",
        "id=7  | precio_compra = null     → valor faltante (NaN)",
        "id=12 | stock = 'texto'          → texto en campo numérico",
        "id=14 | proveedor = null         → proveedor desconocido",
        "id=20 | stock = -30              → stock negativo (lógicamente inválido)",
    ],
    'TXT (IDs 21-40)': [
        "id=27 | precio_compra = 'N/A'   → texto en campo numérico",
        "id=29 | precio_compra = ''      → campo vacío",
        "id=37 | precio_venta_publico = 'error' → texto en campo numérico",
    ],
    'XML (IDs 41-60)': [
        "id=44 | precio_compra = 'error' → texto en campo numérico",
        "id=46 | stock = -10             → stock negativo",
        "id=47 | precio_compra = ''      → campo vacío",
        "id=48 | stock = 'texto'         → texto en campo numérico",
        "id=51 | proveedor = ''          → campo vacío",
        "id=54 | precio_compra = 'N/A'  → texto en campo numérico",
    ]
}
total =0
for fuente, lista in errores.items():
    print(f'\n{'='*80}')
    print(f'Errores en {fuente}:')
    print(f'{'-'*80}')
    for i in lista:
        print(f'Err {i}')
        total +=1
print(f'\n{'-'*80}')
print(f'Total de errores detectados: {total}')
print(f'{'='*80}')


Errores en JSON (IDs 1-20):
--------------------------------------------------------------------------------
Err id=4  | precio_compra = 'error'  → texto en campo numérico
Err id=7  | precio_compra = null     → valor faltante (NaN)
Err id=12 | stock = 'texto'          → texto en campo numérico
Err id=14 | proveedor = null         → proveedor desconocido
Err id=20 | stock = -30              → stock negativo (lógicamente inválido)

Errores en TXT (IDs 21-40):
--------------------------------------------------------------------------------
Err id=27 | precio_compra = 'N/A'   → texto en campo numérico
Err id=29 | precio_compra = ''      → campo vacío
Err id=37 | precio_venta_publico = 'error' → texto en campo numérico

Errores en XML (IDs 41-60):
--------------------------------------------------------------------------------
Err id=44 | precio_compra = 'error' → texto en campo numérico
Err id=46 | stock = -10             → stock negativo
Err id=47 | precio_compra = ''      → campo vacío


In [227]:
for nombre, df in [('JSON', df_json), ('TXT', df_txt), ('XML', df_xml)]:
    nulos = df.isnull().sum().sum()
    vacios = (df =='').sum().sum()
    print(f'{nombre} - Nan/null: {nulos} - cadenas vacias: {vacios}')


JSON - Nan/null: 0 - cadenas vacias: 0
TXT - Nan/null: 2 - cadenas vacias: 0
XML - Nan/null: 3 - cadenas vacias: 0


### A-6. Concatenar con `pd.concat()` y exportar con el metodo to_csv()

`pd.concat()` apila DataFrames verticalmente. `ignore_index=True` reinicia el índice.

| Función | Uso |
|---------|-----|
| `pd.concat()` | Apilar DataFrames con **misma estructura** |

In [228]:
df_inventario = pd.concat([df_json, df_txt, df_xml], ignore_index=True)
print(f'Filas concatenados: {len(df_inventario)} (20 + 20 + 20 = 60)')
print(f'Columnas: {list(df_inventario.columns)}\n')
print(f'{'-'*100}')

print(f'Primeras filas: {df_inventario.head(3)}\n')
print(f'{'-'*100}')
print(f'Ultimas filas: {df_inventario.tail(3)}\t')
print(f'{'-'*100}')



Filas concatenados: 60 (20 + 20 + 20 = 60)
Columnas: ['id', 'nombre', 'precio_compra', 'categoria', 'stock', 'proveedor', 'precio_venta_publico']

----------------------------------------------------------------------------------------------------
Primeras filas:   id                            nombre        precio_compra  categoria stock  \
0  1                Bolígrafo BIC Azul  0.35000000000000003  Escritura   500   
1  2  Cuaderno Universitario 100 hojas                  1.8  Cuadernos   300   
2  3            Lápiz HB Faber-Castell                  0.2  Escritura   800   

            proveedor precio_venta_publico  
0  DistribuPapel S.A.                 0.75  
1       Norma Ecuador                  3.5  
2  DistribuPapel S.A.                 0.45  

----------------------------------------------------------------------------------------------------
Ultimas filas:     id                         nombre precio_compra     categoria stock  \
57  58      Separadores de Colores x5      

Exportar el dataframe concatenado con `to_csv()` con index=False  como argumento adicional, evitar escribir el índice numérico como columna extra

In [229]:
df_inventario.to_csv('inventario_completo.csv ', index=False)
print('Se exporto el inventario_completo.csv concatenado correctamente')
print(f'Total de registros: {len(df_inventario)}')

verificacion = pd.read_csv('inventario_completo.csv')
print(f'Archivo {verificacion.shape[0]} filas y {verificacion.shape[1]} columnas')
verificacion.head()



Se exporto el inventario_completo.csv concatenado correctamente
Total de registros: 60
Archivo 60 filas y 7 columnas


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,1,Bolígrafo BIC Azul,0.35000000000000003,Escritura,500,DistribuPapel S.A.,0.75
1,2,Cuaderno Universitario 100 hojas,1.8,Cuadernos,300,Norma Ecuador,3.5
2,3,Lápiz HB Faber-Castell,0.2,Escritura,800,DistribuPapel S.A.,0.45
3,4,Resaltador Amarillo,error,Escritura,150,OfficeMax Ecuador,1.2
4,5,Carpeta Manila A4,0.15,Archivo,1000,PapelPro Cía. Ltda.,0.3


---
---
## PARTE B: Series, DataFrames, `loc` e `iloc` — Catálogo de Netflix

Trabajaremos con el archivo `netflix.csv`, un dataset real con el catálogo de contenidos de Netflix.

| Columna | Descripción |
|---------|------------|
| `show_id` | ID único del título |
| `type` | `Movie` o `TV Show` |
| `title` | Nombre del título |
| `director` | Director(es) |
| `cast` | Elenco principal |
| `country` | País de producción |
| `date_added` | Fecha de añadido a Netflix |
| `release_year` | Año de estreno |
| `rating` | Clasificación de audiencia |
| `duration` | Duración (minutos o temporadas) |
| `listed_in` | Géneros/categorías |
| `description` | Sinopsis |

### B-1. Carga el dataset y realiza una primera inspección

In [230]:
df_netflix = pd.read_csv('netflix.csv')

print(f'\nForma: {df_netflix.shape[0]} fiilas y {df_netflix.shape[1]} columnas')
print(f'{'-'*100} ')
print(f'Primeras 5 filas  \n{df_netflix.head()}')
print(f'\n{'='*100} ')
print(f'Tipo de dato \n{df_netflix.dtypes}')
print(f'\n{'='*100} ')
print(f'Valores nulos \n{df_netflix.isnull().sum()}')
print(f'\n{'='*100} ')
print(f'Estadistica general: \n{df_netflix.describe()}')
print(f'\n{'='*100} ')


Forma: 6137 fiilas y 15 columnas
---------------------------------------------------------------------------------------------------- 
Primeras 5 filas  
         id                                title   type  \
0  ts300399  Five Came Back: The Reference Films   SHOW   
1   tm82169                                Rocky  MOVIE   
2   tm17823                               Grease  MOVIE   
3  tm191099                            The Sting  MOVIE   
4   tm69975                             Rocky II  MOVIE   

                                         description  release_year  \
0  This collection includes 12 World War II-era p...          1945   
1  When world heavyweight boxing champion, Apollo...          1976   
2  Australian good girl Sandy and greaser Danny f...          1978   
3  A novice con man teams up with an acknowledged...          1973   
4  After Rocky goes the distance with champ Apoll...          1979   

  age_certification  runtime                                 genres  

---
### B-2. ¿Qué es una Series?

Una **Series** es un arreglo **unidimensional** etiquetado. Cada elemento tiene un **valor** y un **índice**.

```python
# Desde una lista
s = pd.Series([10, 20, 30])

# Desde un diccionario (claves = índice)
s = pd.Series({'a': 10, 'b': 20, 'c': 30})
```

| Atributo | Descripción |
|----------|------------|
| `.values` | Datos como array NumPy |
| `.index` | Etiquetas del índice |
| `.dtype` | Tipo de dato |
| `.name` | Nombre de la Series |
| `.shape` | Forma (número de elementos,) |

**Cuando accedes a UNA columna de un DataFrame, obtienes una Series:**

### Extraer una columna = una Series del csv de netflix

In [231]:
serie_anio = df_netflix['release_year']

print(f'Tipo: {type(serie_anio)}')
print(f'Nombre de serie: {serie_anio.name}')
print(f'Primeros 5 valores: \n{serie_anio.head()}')

Tipo: <class 'pandas.core.series.Series'>
Nombre de serie: release_year
Primeros 5 valores: 
0    1945
1    1976
2    1978
3    1973
4    1979
Name: release_year, dtype: int64


### Operaciones comunes sobre Series

In [232]:

print('Titulos posteriores a 1945 ')
serie_filtrada = serie_anio[serie_anio > 1945]
print(f'Total: {len(serie_filtrada)} titulos\n')

print(f'{'-'*70}')
print(f'Atributos de las series:')
print(f'Tipo de dato       -> .dtype  : {serie_anio.dtype}')
print(f'Dimensiones        -> .shape  : {serie_anio.shape}')
print(f'Nombre             -> .name   : {serie_anio.name}')
print(f'Cantidad           -> .size   : {serie_anio.size}')
print(f'primeros 5 indices -> .index  : {list(serie_anio.index[:5])}')
print(f'primeros 5 valores -> .values : {serie_anio.values[:5]}')
print(f'{'-'*70}')

print("Años más repetidos:")
print(serie_anio.value_counts().head())

print(f'{'-'*70}')
print("Cantidad total de datos :", serie_anio.count())
print("Valores unicos          :", serie_anio.nunique())

Titulos posteriores a 1945 
Total: 6136 titulos

----------------------------------------------------------------------
Atributos de las series:
Tipo de dato       -> .dtype  : int64
Dimensiones        -> .shape  : (6137,)
Nombre             -> .name   : release_year
Cantidad           -> .size   : 6137
primeros 5 indices -> .index  : [0, 1, 2, 3, 4]
primeros 5 valores -> .values : [1945 1976 1978 1973 1979]
----------------------------------------------------------------------
Años más repetidos:
release_year
2022    915
2021    868
2020    829
2019    788
2018    743
Name: count, dtype: int64
----------------------------------------------------------------------
Cantidad total de datos : 6137
Valores unicos          : 62


### Obten las estadísticas de la columna 'release_year', count, min, max, mean, median, std

In [233]:

print(f'{'-'*20}Estadisticas de release_year{'-'*20}')
print(f'Total de registros  -> count  : {serie_anio.count()}')
print(f'Año mas antiguo     -> min    : {serie_anio.min()}')
print(f'Año mas actual      -> max    : {serie_anio.max()}')
print(f'Promedio            -> mean   : {serie_anio.mean()}')
print(f'Mediana             -> mean   : {serie_anio.median()}')
print(f'Desviacion Estandar -> std    : {serie_anio.std()}')
print(f'{'-'*68}')


--------------------Estadisticas de release_year--------------------
Total de registros  -> count  : 6137
Año mas antiguo     -> min    : 1945
Año mas actual      -> max    : 2023
Promedio            -> mean   : 2017.3718429199935
Mediana             -> mean   : 2019.0
Desviacion Estandar -> std    : 6.6036201409680695
--------------------------------------------------------------------


### Obten el Top 10 países con más contenido producido

In [234]:
top_paises = df_netflix ['production_countries'].value_counts().head(10)
print(f'{'-'*10}Top 10 paises con mas contenido producido{'-'*10}')
for i, (pais, cantidad) in enumerate(top_paises.items(),1):
    print(f'{i}. {pais} - {cantidad} titulos')

----------Top 10 paises con mas contenido producido----------
1. ['US'] - 1981 titulos
2. ['IN'] - 633 titulos
3. ['JP'] - 278 titulos
4. ['KR'] - 259 titulos
5. ['GB'] - 235 titulos
6. ['ES'] - 177 titulos
7. [] - 176 titulos
8. ['FR'] - 121 titulos
9. ['MX'] - 115 titulos
10. ['BR'] - 104 titulos


### Obten el Top 10 de series/peliculas mas vistas

In [235]:
top_10 =(
    df_netflix[['title', 'type', 'release_year', 'tmdb_popularity']]
    .dropna(subset=['tmdb_popularity'])
    .sort_values('tmdb_popularity', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print(f'{'-'*20}Top de series/peliculas mas vistas{'-'*20}')
print(top_10.to_string(index=True))

--------------------Top de series/peliculas mas vistas--------------------
                      title   type  release_year  tmdb_popularity
0                 The Flash   SHOW          2014         1078.637
1                  Triptych   SHOW          2023         1005.232
2            Grey's Anatomy   SHOW          2005         1003.948
3                 Wednesday   SHOW          2022          833.087
4                    Narvik  MOVIE          2022          779.484
5               Lesson Plan  MOVIE          2022          755.987
6          The Walking Dead   SHOW          2010          656.772
7  The Lying Life of Adults   SHOW          2023          626.158
8                    JUNG_E  MOVIE          2023          598.277
9             The Blacklist   SHOW          2013          592.555


---
### B-3. Acceso con `loc` (por etiqueta)

**`loc`** selecciona datos por **etiqueta** del índice y **nombre** de columna.

```python
df.loc[etiqueta_fila]                    # Una fila completa
df.loc[etiqueta_fila, 'columna']         # Un valor específico
df.loc[inicio:fin]                       # Rango de filas (INCLUSIVO)
df.loc[condicion]                        # Filtrado booleano
df.loc[condicion, ['col1', 'col2']]      # Filtrado + columnas
```

> Con `loc`, los rangos son **INCLUSIVOS** en ambos extremos: `loc[2:5]` incluye 2, 3, 4 **y 5**.

In [236]:
# Genera un ejemplo de Acceso a una fila completa por su etiqueta de índice
print(f'Loc -> Acceso al indice 0: \n{df_netflix.loc[0]}')

Loc -> Acceso al indice 0: 
id                                                               ts300399
title                                 Five Came Back: The Reference Films
type                                                                 SHOW
description             This collection includes 12 World War II-era p...
release_year                                                         1945
age_certification                                                   TV-MA
runtime                                                                51
genres                                                  ['documentation']
production_countries                                               ['US']
seasons                                                               1.0
imdb_id                                                               NaN
imdb_score                                                            NaN
imdb_votes                                                            NaN
tmdb_popul

In [237]:
# Filtrado con MÚLTIPLES condiciones
# Obten el listado de Películas de Estados Unidos producidas desde 2018
peliculas_2018 = df_netflix.loc[
    (df_netflix['type'] == 'MOVIE') &
    (df_netflix['production_countries'].str.contains('US', na=False)) &
    (df_netflix['release_year'] >= 2018),
    ['title', 'release_year', 'imdb_score', 'runtime']
]
print(f"Peliculas de EE.UU. desde 2018: {len(peliculas_2018)} títulos\n")
print(peliculas_2018.sort_values('release_year', ascending=False).head(10).to_string(index=False))

Peliculas de EE.UU. desde 2018: 887 títulos

                          title  release_year  imdb_score  runtime
      Sommore: Queen Chandelier          2023         6.1       69
The Hatchet Wielding Hitchhiker          2023         6.2       87
             Your Place or Mine          2023         5.6       99
                     You People          2023         5.5      117
         Luther: The Fallen Sun          2023         7.0      129
           Pamela, A Love Story          2023         7.2      113
                    True Spirit          2023         6.8      109
  Chris Rock: Selective Outrage          2023         7.1       69
     Jim Jefferies: High n' Dry          2023         7.1       68
                       Dog Gone          2023         6.0       95


In [238]:
# Filtrado con isin() para múltiples valores
# Filtra todas las filas cuyo Contenido tenga como origen Latinoamérica
paises_latam = ['Mexico', 'Argentina', 'Colombia', 'Chile', 'Brazil', 'Peru']

contenido_latam = df_netflix[ df_netflix['production_countries'].isin(paises_latam)]

print(f'Contenido de Latinoamérica: {len(contenido_latam)} titulos')
print()
print(contenido_latam[['title', 'type', 'production_countries', 'release_year']].head(10).to_string(index=False))

Contenido de Latinoamérica: 0 titulos

Empty DataFrame
Columns: [title, type, production_countries, release_year]
Index: []


In [239]:
# str.contains(): filtrado por texto parcial en columna
# Títulos que contengan 'Love' en el nombre
titulos_love = df_netflix.loc[
    df_netflix['title'].str.contains('Love', case=False, na=False),
    ['title', 'type', 'release_year', 'imdb_score']
]

print(f"Títulos que contienen 'Love': {len(titulos_love)}")
print()
print(titulos_love.sort_values('imdb_score', ascending=False).head(10).to_string(index=False))

Títulos que contienen 'Love': 153

                       title  type  release_year  imdb_score
        Love on the Spectrum  SHOW          2019         8.6
Love Between Fairy and Devil  SHOW          2022         8.6
        Love, Death & Robots  SHOW          2019         8.4
                  First Love  SHOW          2022         8.4
                Eternal Love  SHOW          2017         8.3
               Ashes of Love  SHOW          2018         8.3
         Just Between Lovers  SHOW          2017         8.2
   Love on the Spectrum U.S.  SHOW          2022         8.2
                  Love Today MOVIE          2022         8.1
                    Lovesick  SHOW          2014         8.0


### B-4. Ya que tienes acceso a los datos de Netflix, en base a tu curiosidad, usando pandas obten al menos 3 datos de interes adicionales, que te gustaria conocer.

In [240]:
contenido_por_anio = df_netflix['release_year'].value_counts().sort_index()

anio_pico = contenido_por_anio.idxmax()
cantidad_pico = contenido_por_anio.max()

anio_min = df_netflix['release_year'].min()
anio_max = df_netflix['release_year'].max()
total_anios = anio_max - anio_min + 1

print('Cantidad de contenido publicado por cada año')
print(contenido_por_anio.to_string())

print()
print(f'Rango de años existente: {anio_min} al {anio_max}')
print(f'Total de años registrados: {total_anios}')
print()
print(f'El año con mas contenido lanzado: {anio_pico} con {cantidad_pico} titulos')

Cantidad de contenido publicado por cada año
release_year
1945      1
1954      2
1956      1
1958      1
1962      1
1963      1
1966      1
1969      2
1970      1
1971      3
1972      2
1973      2
1974      1
1975      3
1976      2
1977      2
1978      3
1979      3
1980      3
1981      3
1982      5
1983      3
1984      2
1985      4
1986      9
1987      3
1988      6
1989      7
1990      6
1991      8
1992      4
1993     10
1994     10
1995     11
1996      8
1997      8
1998     19
1999     14
2000     11
2001     18
2002     18
2003     19
2004     35
2005     29
2006     39
2007     35
2008     43
2009     55
2010     53
2011     61
2012     84
2013    103
2014    137
2015    195
2016    312
2017    475
2018    743
2019    788
2020    829
2021    868
2022    915
2023     97

Rango de años existente: 1945 al 2023
Total de años registrados: 79

El año con mas contenido lanzado: 2022 con 915 titulos


In [241]:
series = df_netflix[df_netflix['type'] == 'SHOW'].copy()

series_top = series.sort_values(by='seasons', ascending=False)

print('Series con mayor cantidad de temporadas')
print(series_top[['title', 'seasons', 'release_year']].head(10).to_string(index=False))

Series con mayor cantidad de temporadas
           title  seasons  release_year
        Survivor     44.0          2000
The Amazing Race     34.0          2001
  The Real World     33.0          1992
        Top Gear     33.0          2002
American Pickers     31.0          2010
   Power Rangers     29.0          1993
         Pokémon     25.0          1997
Thomas & Friends     24.0          1984
    Intervention     24.0          2005
     Big Brother     24.0          2000


In [242]:
conteo_tipo = df_netflix['type'].value_counts()
print('Cantidad peliculas y Series')
for tipo, cantidad in conteo_tipo.items():
    pct = cantidad / len(df_netflix) * 100
    print(f"  {tipo:<6}: {cantidad:>5} titulos  ({pct:.1f}%)")

Cantidad peliculas y Series
  MOVIE :  3831 titulos  (62.4%)
  SHOW  :  2306 titulos  (37.6%)


---
### B-5. Acceso con `iloc` (por posición entera)

**`iloc`** selecciona datos por **posición numérica** (como índices de listas en Python).

```python
df.iloc[posicion]                          # Una fila
df.iloc[pos_fila, pos_columna]             # Un valor
df.iloc[inicio:fin]                        # Rango (EXCLUSIVO al final)
df.iloc[[0, 5, 10]]                        # Posiciones específicas
df.iloc[::n]                               # Cada n filas
```

> Con `iloc`, el rango es **EXCLUSIVO** al final: `iloc[2:5]` incluye posiciones 2, 3, 4 pero **NO 5**.

### Diferencia clave `loc` vs `iloc`:

| | `loc` | `iloc` |
|--|-------|--------|
| **Acceso** | Por **etiqueta** | Por **posición** |
| **Rango** | **Inclusivo** | **Exclusivo** |
| **Columnas** | Por nombre | Por número (0, 1, 2...) |
| **Filtrado** |  Booleano | No directo |

In [243]:
# Obten la Primera fila (posición 0) usando loc
print(f'Primera fila: {df_netflix.iloc[0]}')

Primera fila: id                                                               ts300399
title                                 Five Came Back: The Reference Films
type                                                                 SHOW
description             This collection includes 12 World War II-era p...
release_year                                                         1945
age_certification                                                   TV-MA
runtime                                                                51
genres                                                  ['documentation']
production_countries                                               ['US']
seasons                                                               1.0
imdb_id                                                               NaN
imdb_score                                                            NaN
imdb_votes                                                            NaN
tmdb_popularity         

In [244]:
# Última fila (índice negativo, como listas de Python) usando loc
print(f'Ultima fila {df_netflix.iloc[-1]}')

Ultima fila id                                                               tm561709
title                                                     Going to Heaven
type                                                                MOVIE
description             A story about young boy sultan, 11 years old l...
release_year                                                         2023
age_certification                                                     NaN
runtime                                                                90
genres                                                         ['family']
production_countries                                               ['AE']
seasons                                                               NaN
imdb_id                                                         tt4623222
imdb_score                                                            7.0
imdb_votes                                                           40.0
tmdb_popularity           

In [245]:
# Valor específico: fila 10, columna 2 (title) usando loc
valor = df_netflix.loc[10, 'type']
titulo = df_netflix.loc[10, 'title']

print(f'Título: {titulo}')
print(f'Tipo: {valor}')

Título: Heroes
Tipo: MOVIE


In [246]:
# Rango de filas (EXCLUSIVO: posiciones 2, 3, 4 — NO incluye 5) usando loc
print(df_netflix.loc[2:4, ['title', 'type', 'release_year']].to_string(index=False))

    title  type  release_year
   Grease MOVIE          1978
The Sting MOVIE          1973
 Rocky II MOVIE          1979


In [247]:
# Usando loc , obten los 10 títulos más antiguos en Netflix
# Combinamos sort_values con iloc(Ejemplo base: netflix_ordenado = df_netflix.sort_values("release_year", ascending=True))
print('=== 10 títulos más antiguos en Netflix ===')
netflix_ordenado = df_netflix.sort_values('release_year', ascending=True).reset_index(drop=True)

diez_mas_antiguos = netflix_ordenado.loc[0:9, ['title', 'type', 'release_year']]

print('10 titulos mas antiguos en Netflix')
print(diez_mas_antiguos.to_string(index=False))

=== 10 títulos más antiguos en Netflix ===
10 titulos mas antiguos en Netflix
                              title  type  release_year
Five Came Back: The Reference Films  SHOW          1945
                    The Blazing Sun MOVIE          1954
                    White Christmas MOVIE          1954
                        Dark Waters MOVIE          1956
                      Cairo Station MOVIE          1958
                          Professor MOVIE          1962
             Saladin the Victorious MOVIE          1963
                           Amrapali MOVIE          1966
                             Prince MOVIE          1969
       Monty Python's Flying Circus  SHOW          1969


---
---
## PARTE C: Procesamiento de Texto — *Dracula* (Bram Stoker)

En esta sección crear las cajas pertinentes a fin de realizar las siguientes actividades sobre el libro de Drácula:
1. Descargar texto desde una URL con `urllib`, crear una variable time, que contabilize el tiempo desde que se inicia la descarga.
2. Eliminar encabezados y pies de página propios de Project Gutenberg.
3. Eliminar signos de puntuación, números y caracteres especiales con `re`.
4. Procesar texto: minúsculas, tokenizar, filtrar.
5. Contar frecuencias con `collections.Counter`.
6. Mostrar el top 20 de las palabras mas frecuentes.
7. Guardar resultados en un archivo de texto(txt), donde se muestre el top 20 de palabras mas frecuentes y su cantidad de incidencias, asi como el tiempo total demorado desde que se descarga el libro, hasta que se cuenta las palabras mas frecuentes.

Usaremos **"Dracula"** de Bram Stoker, disponible en [Project Gutenberg](https://www.gutenberg.org/ebooks/345).  
https://www.gutenberg.org/cache/epub/345/pg345.txt



In [248]:
#Desarrollo del estudiante
#1. Descargar Texto

url_pelicula = 'https://www.gutenberg.org/cache/epub/345/pg345.txt'

print('Descargado el libro')
Ini_descarga = time.time()
response = urllib.request.urlopen(url_pelicula)
texto = response.read().decode('utf-8')

print(f'Tiempo de Descarga: {time.time()- Ini_descarga}\n')
print(f'Tamaño de texto: {len(texto)} caracteres')
print(f'Mostrar los 300 primeros caracteres \n{texto[:300]}')


Descargado el libro
Tiempo de Descarga: 31.291407585144043

Tamaño de texto: 881061 caracteres
Mostrar los 300 primeros caracteres 
﻿The Project Gutenberg eBook of Dracula
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License inc


In [249]:
#2. Eliminar encabezados y pie de paginas
marcador_inicio = '*** START OF THE PROJECT GUTENBERG EBOOK'
marcador_fin = '*** END OF THE PROJECT GUTENBERG EBOOK'

pos_inicio = texto.find(marcador_inicio)
pos_fin = texto.find(marcador_fin)
print(f'Marcador de inicio encontrado en: {pos_inicio}')
print(f'Marcador de fin encontrado en: {pos_fin}')
print(f'Cantidad de caracteres: {len(texto)}')
texto_limp = texto[texto.index('\n', pos_inicio)+1 : pos_fin].strip()
print(f'Caracteres del libro: {len(texto_limp):,}') 
print(f'Se eliminaron: {len(texto)- len(texto_limp):,} caracteres de encabezazdo/pie\n')
print(f'Primeros 200 caracteres \n{texto_limp[:200]}') 
print(f'Ultimos 200 caracteres \n{texto_limp[-200:]}') 



Marcador de inicio encontrado en: 812
Marcador de fin encontrado en: 862271
Cantidad de caracteres: 881061
Caracteres del libro: 861,355
Se eliminaron: 19,706 caracteres de encabezazdo/pie

Primeros 200 caracteres 
DRACULA

                                  _by_

                              Bram Stoker

                        [Illustration: colophon]

                                NEW YORK

      
Ultimos 200 caracteres 
LADE AMALGAMATION

THE SAFETY PIN

THE SECRET WAY

THE VALLEY OF HEADSTRONG MEN

_Ask for Complete free list of G. & D. Popular Copyrighted Fiction_

GROSSET & DUNLAP, _Publishers_, NEW YORK


In [250]:
#3. Eliminar signos de puntuación
text_sin_Puntuacion = re.sub(r'[^a-zA-Z\s]', ' ',texto_limp)



In [251]:
#4. Procesar texto: 
text_minuscl = text_sin_Puntuacion.lower()
token = text_minuscl.split()
long_min = 3
palabra_filtr = [p for p in token if len(p) >= long_min]
print(f'Total tokens: {len(token):,}')
print(f'Total palabrasa filtradas: {long_min} letras: {len(palabra_filtr):,}')
print(f'Plabras filtradas \n{palabra_filtr[:20]}')

Total tokens: 163,401
Total palabrasa filtradas: 3 letras: 121,929
Plabras filtradas 
['dracula', 'bram', 'stoker', 'illustration', 'colophon', 'new', 'york', 'grosset', 'dunlap', 'publishers', 'copyright', 'the', 'united', 'states', 'america', 'according', 'act', 'congress', 'bram', 'stoker']


In [252]:
#5. Contar frecuencias
c_frecuencia = Counter(palabra_filtr)

In [253]:
#6. Mostrar top 20 de las palabras mas frecuentes
top_20 = c_frecuencia.most_common(20)
inicio = time.time()
print(f'{'-'*20}Top 20 de las palabras mas frecuentes{'-'*20}')
print(f'{"N.-":<4} {"Palabra":<20} {"Frecuencia"}')
for i , (palabra, frecuencia) in enumerate(top_20,1):
    print(f'{i:<4} {palabra:<20} {frecuencia}')

print(f'Palabras unicas: {len(c_frecuencia):,}')




--------------------Top 20 de las palabras mas frecuentes--------------------
N.-  Palabra              Frecuencia
1    the                  7915
2    and                  5907
3    that                 2488
4    was                  1881
5    for                  1538
6    his                  1472
7    you                  1413
8    not                  1401
9    with                 1287
10   all                  1169
11   but                  1071
12   have                 1061
13   her                  1061
14   had                  1039
15   him                  954
16   she                  816
17   there                781
18   when                 777
19   which                663
20   this                 629
Palabras unicas: 9,181


In [257]:
#7. Guardar ressultados en archivo de texto
inicio_t = time.time()
Tiempo_t = time.time() - inicio_t
archivo = 'Dracula_Top.txt'
with open (archivo, 'w', encoding='utf-8') as f:
    f.write('Texto : Dracula - Autor: Bram stoker\n')
    f.write('Fuente: Project Gutenberg\n')
    f.write('-'*50+'\n')
    f.write('\tTop de 20 palabras mas frecuentes\n')
    f.write('-'*50+'\n')
    f.write(f'{'N.-':<6} {'Palabra':<10} {'Ocurrencias':<15}\n')
    for i, (palabra, frecuencia) in enumerate(top_20, 1):
        f.write(f'{i:<6} {palabra:<15} {frecuencia:<15,}\n')
    f.write('\n'+'-' * 50 + '\n')
    f.write(f'Tiempo total :{Tiempo_t:.2f} segundos\n')
    f.write(f'Palabras analizadas :{Tiempo_t:.2f} segundos\n')
    f.write('-' * 50 + '\n')
print(f'Archivo generado con exito: {archivo}')

print(f'\n{'='*18}Archivo Guardado{'='*18}')
with open (archivo, 'r', encoding='utf-8') as f:
    print(f.read())
print('='*50)


Archivo generado con exito: Dracula_Top.txt

==================Archivo Guardado==================
Texto : Dracula - Autor: Bram stoker
Fuente: Project Gutenberg
--------------------------------------------------
	Top de 20 palabras mas frecuentes
--------------------------------------------------
N.-    Palabra    Ocurrencias    
1      the             7,915          
2      and             5,907          
3      that            2,488          
4      was             1,881          
5      for             1,538          
6      his             1,472          
7      you             1,413          
8      not             1,401          
9      with            1,287          
10     all             1,169          
11     but             1,071          
12     have            1,061          
13     her             1,061          
14     had             1,039          
15     him             954            
16     she             816            
17     there           781            
18   